# Atomic electronic structure: Python example

A numerical reimplementation of the MATLAB coursework approach. Default: spherical, spin-unpolarised, exchange-only X-alpha with alpha=1. **These are model results, not experimental reference values or a verified reproduction of the MATLAB output.**

Run all cells from a fresh kernel. No historical notebook functions are needed.

In [1]:
from pathlib import Path
import sys
import numpy as np

module_dir = Path.cwd() if (Path.cwd() / "modules.py").is_file() else Path.cwd() / "python"
if not (module_dir / "modules.py").is_file():
    raise RuntimeError("Start Jupyter from the project root or python directory.")
sys.path.insert(0, str(module_dir.resolve()))
from modules import AtomicSolver, CONFIGURATIONS

## 1. Nuclear-only analytic check

For hydrogen without electron-electron terms, the exact levels are -1/(2n²) Hartree. This is different from the self-interacting one-electron X-alpha approximation.

In [2]:
hydrogen = AtomicSolver(1, rmax=60)
energies, coefficients, residual = hydrogen.hydrogenic(l=0, count=3)
exact = -0.5 / np.arange(1, 4)**2
print(" n       computed / Ha       exact / Ha       absolute error")
for n, energy, reference in zip(range(1, 4), energies, exact):
    print(f"{n:2d} {energy:20.10f} {reference:16.10f} {abs(energy-reference):16.3e}")
print("Relative eigenproblem residual:", residual)

 n       computed / Ha       exact / Ha       absolute error
 1        -0.5000000000    -0.5000000000        3.443e-11
 2        -0.1250000000    -0.1250000000        2.364e-11
 3        -0.0555555555    -0.0555555556        7.086e-11
Relative eigenproblem residual: 1.3066913696478694e-12


## 2. Neutral helium and helium ion

The two calculations deliberately have different electron occupations. The default alpha=1 matches the exchange-potential coefficient in the uploaded MATLAB code. It does not give exact physical helium energies. Set alpha=2/3 for Dirac exchange.

In [3]:
solver = AtomicSolver(Z=2, alpha=1.0)
neutral = solver.solve(CONFIGURATIONS["He"][1])
ion = solver.solve(CONFIGURATIONS["He+"][1])
for label, result in [("He", neutral), ("He+", ion)]:
    print(f"{label}: E={result.total_energy:.10f} Ha, N={result.electron_count:.10f}, "
          f"iterations={result.iterations}, residual={result.density_residual:.3e}")
print("Model ionisation energy / Ha:", ion.total_energy - neutral.total_energy)

He: E=-3.1701122438 Ha, N=2.0000000000, iterations=43, residual=9.337e-09
He+: E=-2.0133866362 Ha, N=1.0000000000, iterations=41, residual=8.231e-09
Model ionisation energy / Ha: 1.1567256075306402


## 3. Inspect normalisation and SCF history

The radial density q=4πr²ρ integrates to the electron number, using the quadrature weights. Radius never equals zero at these integration nodes.

In [4]:
print("Integrated electron count:", neutral.weights @ neutral.radial_density)
print("Density finite:", bool(np.all(np.isfinite(neutral.density))))
print("Last five SCF steps:")
for step in neutral.history[-5:]:
    print(step)

Integrated electron count: 1.9999999999999998
Density finite: True
Last five SCF steps:
{'iteration': 39, 'energy_hartree': -3.1701122437510225, 'density_residual': 3.7039087696241055e-08, 'energy_change': 1.7763568394002505e-15}
{'iteration': 40, 'energy_hartree': -3.170112243751028, 'density_residual': 2.6224966830737384e-08, 'energy_change': 5.329070518200751e-15}
{'iteration': 41, 'energy_hartree': -3.170112243751022, 'density_residual': 1.85785290815454e-08, 'energy_change': 5.773159728050814e-15}
{'iteration': 42, 'energy_hartree': -3.17011224375102, 'density_residual': 1.3168002951904801e-08, 'energy_change': 2.220446049250313e-15}
{'iteration': 43, 'energy_hartree': -3.170112243751022, 'density_residual': 9.336965252876718e-09, 'energy_change': 2.220446049250313e-15}


## Next examples

From the repository root run `python python/run_atom.py Ne --ion` or `python python/run_atom.py K --ion`. Run `python -m unittest discover -s tests -v` for tests and `python validation/run_validation.py` for the mesh/box/quadrature study. See `validation/REPORT.md` for measured results and limits.